In [44]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
import json
from langchain_core.messages import SystemMessage,HumanMessage, ToolMessage

load_dotenv()

True

In [45]:
model = ChatOpenAI(model="gpt-4o-mini")

In [46]:
with open("bajaj_db.json") as f:
    db = json.load(f)


@tool
def get_details(loan_id):
    "this function will help you to get the loan details  args : loan_id (str): The ID of the loan"
    details = db['loans'].get(loan_id, 'No info for this id is available')
    return details

@tool
def next_emi_date(loan_id):
    """
    This function will help you to get the next EMI date
    args : loan_id (str): The ID of the loan
    """
    details = db['loans'].get(loan_id, {})
    return details.get('next_due_date')

@tool
def get_weather_info(city):
    "This function will return the temp of provided cities"
    if city.lower() == 'mumbai':
        return "34 degree"
    return 'We dont have api acces'


tools = [get_details, next_emi_date, get_weather_info]

model_with_tools = model.bind_tools(tools)

In [47]:
# 
# model.invoke('what is my emi for loan id BFL2024001')

In [ ]:
# prompt= ChatPromptTemplate.from_messages([
#    ('system',''' Your are a expert Financial advisor.'''),
#    ('human','{questions}' )])

# chain = prompt | model_with_tools 

# result = chain.invoke({'questions': 'what is my emi amount for loan id BFL2024001'})

In [ ]:
# result.tool_calls

[{'name': 'get_details',
  'args': {'loan_id': 'BFL2024001'},
  'id': 'call_1xK0AKufTI4e1vsGz2iFUdBU',
  'type': 'tool_call'}]

In [63]:
result = model_with_tools.invoke("what is principle amount  for loan id BFL2024001 when is the my next emi date ")

In [64]:
tool_map = {
    'get_details': get_details,
    'next_emi_date': next_emi_date,
    'get_weather_info': get_weather_info
}


tool_result = []

for tool_call in result.tool_calls:
    tool_name = tool_call['name']
    tool_args = tool_call['args']
    tool_id = tool_call['id']
    test = tool_map[tool_name].invoke(tool_args)
    tool_result.append((tool_id, test))


In [65]:
tool_result

[('call_1BXfeX7E1dVlHo08nwepzOEB',
  {'customer_name': 'Rahul Tiwari',
   'loan_type': 'Personal Loan',
   'principal': 500000,
   'emi': 8450,
   'tenure_months': 72,
   'paid_months': 50,
   'remaining_months': 22,
   'outstanding': 185900,
   'interest_rate': 11.5,
   'next_due_date': '2026-05-05',
   'prepayment_allowed': True,
   'prepayment_charge_pct': 2.0}),
 ('call_nLxweJvxOHU0FCinPtEoPO2r', '2026-05-05')]

In [66]:
system = SystemMessage(content="You are a Bajaj Finance support agent. Use tools for real data.")
user_msg = HumanMessage(content="what is principle amount  for loan id BFL2024001 when is the my next emi date ")
messages = [system,user_msg,result]



In [68]:
messages

[SystemMessage(content='You are a Bajaj Finance support agent. Use tools for real data.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is principle amount  for loan id BFL2024001 when is the my next emi date ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 151, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8b9bd3ab6c', 'id': 'chatcmpl-EO34HbukfXHkktyW9cZGh64oPrNcn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a092-d442-75b2-b662-0760f67f5d21-0', tool_calls=[{'name': 'get_details'

In [60]:
messages

[SystemMessage(content='You are a Bajaj Finance support agent. Use tools for real data.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is principle amount  for loan id BFL2024001 when is the my next emi date ', additional_kwargs={}, response_metadata={}),
 '2026-05-05']

In [69]:
for tool_id, result in tool_result:
    tool_msg = ToolMessage(content=str(result),tool_call_id=tool_id)
    messages.append(tool_msg)

In [70]:
messages

[SystemMessage(content='You are a Bajaj Finance support agent. Use tools for real data.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is principle amount  for loan id BFL2024001 when is the my next emi date ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 151, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_8b9bd3ab6c', 'id': 'chatcmpl-EO34HbukfXHkktyW9cZGh64oPrNcn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0a092-d442-75b2-b662-0760f67f5d21-0', tool_calls=[{'name': 'get_details'

In [57]:
messages

[SystemMessage(content='You are a Bajaj Finance support agent. Use tools for real data.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is principle amount  for loan id BFL2024001 when is the my next emi date ', additional_kwargs={}, response_metadata={}),
 '2026-05-05',
 ToolMessage(content="{'customer_name': 'Rahul Tiwari', 'loan_type': 'Personal Loan', 'principal': 500000, 'emi': 8450, 'tenure_months': 72, 'paid_months': 50, 'remaining_months': 22, 'outstanding': 185900, 'interest_rate': 11.5, 'next_due_date': '2026-05-05', 'prepayment_allowed': True, 'prepayment_charge_pct': 2.0}", tool_call_id='call_17ibz4Aqnf8ns6RkoKug9g1Z'),
 ToolMessage(content='2026-05-05', tool_call_id='call_duTbSqq4pHjZSEDv8gLrXv8H')]

In [71]:
response = model_with_tools.invoke(messages)

In [72]:
response

AIMessage(content='The principal amount for loan ID BFL2024001 is ₹500,000. Your next EMI date is on **May 5, 2026**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 347, 'total_tokens': 381, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1d2403c701', 'id': 'chatcmpl-EO34fQt2AjnIHlAjThlgcYiYMYTno', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a093-3502-7d80-9f79-bb52ee552bcd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 347, 'output_tokens': 34, 'total_tokens': 381, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [26]:
response = model_with_tools.invoke('what is my emi for loan id rweweewr and what is my next 6 month emi schedule date')

In [28]:
response.content

''

In [29]:
response.tool_calls

[{'name': 'get_details',
  'args': {'loan_id': 'rweweewr'},
  'id': 'call_ObXBn0sHUKwxOZ7KtSiBZTvk',
  'type': 'tool_call'},
 {'name': 'next_emi_date',
  'args': {'loan_id': 'rweweewr'},
  'id': 'call_9LeG6lxmoFaZb5NnHTbHN4qu',
  'type': 'tool_call'}]